<a href="https://colab.research.google.com/github/mesata/Store-Sales-Forecasting/blob/main/walmart_sarima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Walmart Weekly Sales Forecasting — Pure SARIMA

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_PATH = '/content/drive/MyDrive/ML final project'

if not os.path.exists(PROJECT_PATH):
    os.makedirs(PROJECT_PATH)
    print(f"it created: {PROJECT_PATH}")

Mounted at /content/drive


In [2]:
%cd {PROJECT_PATH}

/content/drive/.shortcut-targets-by-id/13YZMRWkS2eT6kmRi5e5vzlG7Yl3Vbt3z/ML final project


In [3]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.

In [4]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
import mlflow
import mlflow.sklearn

In [5]:
!pip install dagshub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.0 MB/s eta 0:00:00


In [6]:
import dagshub
dagshub.init(repo_owner='mesata', repo_name='Walmart---Store-Sales-Forecasting', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=3a3911b1-05e8-41d0-b375-f84761299bfe&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=1fdb07f49eed58cfaf7409787d974f8c7217cf00836f3268f2fc79cfe02f2f39




Accessing as mesata

Initialized MLflow to track repo "mesata/Walmart---Store-Sales-Forecasting"

Repository mesata/Walmart---Store-Sales-Forecasting initialized!

In [7]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/train.csv.zip')
features_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/features.csv.zip')
stores_df = pd.read_csv('/content/drive/MyDrive/ML final project/walmart_data/stores.csv')

train_df['Date'] = pd.to_datetime(train_df['Date'])
features_df['Date'] = pd.to_datetime(features_df['Date'])

weekly_sales = (
    train_df.groupby('Date', as_index=False)['Weekly_Sales']
    .sum()
    .sort_values('Date')
    .reset_index(drop=True)
)

holiday_flags = features_df[['Date', 'IsHoliday']].drop_duplicates(subset='Date')
weekly_sales = weekly_sales.merge(holiday_flags, on='Date', how='left')
weekly_sales['IsHoliday'] = weekly_sales['IsHoliday'].fillna(False)

weekly_sales = weekly_sales.set_index('Date')
weekly_sales.index.freq = 'W-FRI'  # Walmart მონაცემები კვირეულია, პარასკევებზე

print(weekly_sales.shape)
weekly_sales.head()

(143, 2)


,Weekly_Sales,IsHoliday
Date,,
2010-02-05,49750740.50,False
2010-02-12,48336677.63,True
2010-02-19,48276993.78,False
2010-02-26,43968571.13,False
2010-03-05,46871470.30,False


In [8]:
VAL_WEEKS = 12

train_series = weekly_sales.iloc[:-VAL_WEEKS]
val_series = weekly_sales.iloc[-VAL_WEEKS:]

print(f"Train: {train_series.index.min()} -> {train_series.index.max()} ({len(train_series)} კვირა)")
print(f"Val:   {val_series.index.min()} -> {val_series.index.max()} ({len(val_series)} კვირა)")

Train: 2010-02-05 00:00:00 -> 2012-08-03 00:00:00 (131 კვირა)
Val:   2012-08-10 00:00:00 -> 2012-10-26 00:00:00 (12 კვირა)


In [9]:
def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [10]:
import mlflow
import mlflow.statsmodels
from statsmodels.tsa.statespace.sarimax import SARIMAX
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

mlflow.set_experiment("Walmart-Sales-Forecasting-SARIMA")

runs_config = [
    {"name": "sarima_1_1_1_52", "order": (1, 1, 1), "seasonal_order": (1, 1, 1, 52)},
    {"name": "sarima_2_1_1_52", "order": (2, 1, 1), "seasonal_order": (1, 0, 1, 52)},
    {"name": "sarima_1_1_0_52", "order": (1, 1, 0), "seasonal_order": (0, 1, 1, 52)},
    {"name": "sarima_2_1_2_52", "order": (2, 1, 2), "seasonal_order": (1, 1, 1, 52)},
]

results = []

for cfg in runs_config:
    with mlflow.start_run(run_name=cfg["name"]):
        mlflow.log_params({
            "model_type": "SARIMA",
            "order": cfg["order"],
            "seasonal_order": cfg["seasonal_order"],
            "val_weeks": VAL_WEEKS,
        })

        model = SARIMAX(
            train_series['Weekly_Sales'],
            order=cfg["order"],
            seasonal_order=cfg["seasonal_order"],
            enforce_stationarity=False,
            enforce_invertibility=False,
        )

        fit_result = model.fit(disp=False, low_memory=True)

        train_pred = fit_result.fittedvalues

        train_wmae = wmae(
            train_series['Weekly_Sales'].values[52:],
            train_pred.values[52:],
            train_series['IsHoliday'].values[52:],
        )

        forecast = fit_result.get_forecast(steps=VAL_WEEKS)
        val_pred = forecast.predicted_mean.values

        val_wmae = wmae(
            val_series['Weekly_Sales'].values,
            val_pred,
            val_series['IsHoliday'].values,
        )

        mlflow.log_metrics({
            "train_wmae": train_wmae,
            "val_wmae": val_wmae,
            "aic": fit_result.aic,
            "bic": fit_result.bic,
        })

        fig, ax = plt.subplots(figsize=(10, 4))
        ax.plot(train_series.index[-30:], train_series['Weekly_Sales'].values[-30:], label="train (last 30w)")
        ax.plot(val_series.index, val_series['Weekly_Sales'].values, label="val actual")
        ax.plot(val_series.index, val_pred, label="val forecast", linestyle="--")
        ax.set_title(cfg["name"])
        ax.legend()

        fig_path = f"/content/{cfg['name']}_forecast.png"
        fig.savefig(fig_path)
        mlflow.log_artifact(fig_path)
        plt.close(fig)

        mlflow.statsmodels.log_model(fit_result, artifact_path="model")

        results.append({**cfg, "train_wmae": train_wmae, "val_wmae": val_wmae})

        avg_val_sales = val_series['Weekly_Sales'].mean()
        val_pct_error = (val_wmae / avg_val_sales) * 100
        print(f"{cfg['name']}: train_wmae={train_wmae:,.2f}, val_wmae={val_wmae:,.2f} (~{val_pct_error:.2f}% error)")

results_df = pd.DataFrame(results)
results_df

2026/07/12 18:50:23 INFO mlflow.tracking.fluent: Experiment with name 'Walmart-Sales-Forecasting-SARIMA' does not exist. Creating a new experiment.
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
2026/07/12 18:50:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


sarima_1_1_1_52: train_wmae=6,065,427.54, val_wmae=678,262.86 (~1.47% error)
🏃 View run sarima_1_1_1_52 at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/10/runs/560f13214115497ebebd9d4a303b295e
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/10


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
2026/07/12 18:51:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


sarima_2_1_1_52: train_wmae=1,533,939.54, val_wmae=1,374,021.74 (~2.97% error)
🏃 View run sarima_2_1_1_52 at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/10/runs/ecf192329b7a4c0f880523cbcc249fbc
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/10


2026/07/12 18:51:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


sarima_1_1_0_52: train_wmae=2,728,009.32, val_wmae=1,378,974.69 (~2.98% error)
🏃 View run sarima_1_1_0_52 at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/10/runs/f8dfdbbae9a049ed9eefe8aebcaf6968
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/10


2026/07/12 18:52:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


sarima_2_1_2_52: train_wmae=91,989,996.63, val_wmae=683,731.71 (~1.48% error)
🏃 View run sarima_2_1_2_52 at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/10/runs/d8c84b278f474370bf5611bb154b53c9
🧪 View experiment at: https://dagshub.com/mesata/Walmart---Store-Sales-Forecasting.mlflow/#/experiments/10


,name,order,seasonal_order,train_wmae,val_wmae
0,sarima_1_1_1_52,"(1, 1, 1)","(1, 1, 1, 52)",6.065428e+06,6.782629e+05
1,sarima_2_1_1_52,"(2, 1, 1)","(1, 0, 1, 52)",1.533940e+06,1.374022e+06
2,sarima_1_1_0_52,"(1, 1, 0)","(0, 1, 1, 52)",2.728009e+06,1.378975e+06
3,sarima_2_1_2_52,"(2, 1, 2)","(1, 1, 1, 52)",9.199000e+07,6.837317e+05
